In [201]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import cv2
import math
import matplotlib.colors as colors

In [202]:
def BC(hist1, hist2):
    hist1 = hist1 / np.sum(hist1)
    hist2 = hist2 / np.sum(hist2)
    BC = np.sum(np.sqrt(hist1 * hist2))
    return BC



In [203]:
# open video file (use VideoCapture, not imread)
vidCapture = cv2.VideoCapture('case.mp4')
# vidCapture = cv2.VideoCapture('rubics.mp4')
# vidCapture = cv2.VideoCapture('walk.mp4')
# vidCapture = cv2.VideoCapture('walksit.mp4')
# vidCapture = cv2.VideoCapture('test_person.mp4')

# ret, frame = vidCapture.read()
# x = 70
# y = 150
# w = 230
# h = 180
# track_window = (x, y, w, h)

# cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
# plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
# plt.show()

# vidCapture = cv2.VideoCapture('case.mp4')
ret, frame = vidCapture.read()

# Let user drag a rectangle around the object
x, y, w, h = cv2.selectROI("Select Object", frame, fromCenter=False)

cv2.destroyWindow("Select Object")

track_window = (x, y, w, h)


Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!


In [ ]:
roi = frame[y:y+h, x:x+w]
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
hist_target = cv2.calcHist([hsv_roi], [0], None, [180], [0, 180]).astype(np.float32)
cv2.normalize(hist_target, hist_target, 0, 1, cv2.NORM_MINMAX)

initial_prob_map = cv2.calcBackProject([cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)], [0], hist_target, [0,180], 1)
m00_0 = np.sum(initial_prob_map[y:y+h, x:x+w])

r1 = w / math.sqrt(m00_0)
r2 = h / math.sqrt(m00_0)


plt.plot(hist_target)
plt.show()


error: OpenCV(4.12.0) /Users/xperience/GHA-Actions-OpenCV/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


: 

In [ ]:
max_iters = 100

while True:
    ret, frame = vidCapture.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    prob_map = cv2.calcBackProject([hsv], [0], hist_target, [0,180], 1)
    prob_map = prob_map.astype(np.float32)   # DO NOT NORMALIZE TO 0–1

    for _ in range(max_iters):
        x, y, w, h = track_window

        window = prob_map[y:y+h, x:x+w]

        m00 = np.sum(window)
        if m00 < 1e-3:
            break

        # moments in window coordinates
        M01 = np.sum(np.arange(w)[None, :] * window)    # x-weighted
        M10 = np.sum(np.arange(h)[:, None] * window)    # y-weighted

        xc = M01 / m00
        yc = M10 / m00


        # correct centroid → global frame
        new_x = int(x + xc - w/2)
        new_y = int(y + yc - h/2)

        height, width = frame.shape[:2]
        new_x = max(0, min(new_x, width - w))
        new_y = max(0, min(new_y, height - h))

        if abs(new_x - x) < 1 and abs(new_y - y) < 1:
            break

        track_window = (new_x, new_y, w, h)
    
    # new_w = int(r1 * math.sqrt(m00))
    # new_h = int(r2 * math.sqrt(m00))

    # x, y, _, _ = track_window

    # # Clamp width/height so box stays inside image
    # H, W = frame.shape[:2]

    # # Adjust width/height if they overflow
    # if x + new_w > W:
    #     new_w = W - x
    # if y + new_h > H:
    #     new_h = H - y

    # # Enforce minimum window size (avoid collapse)
    # new_w = max(new_w, 10)
    # new_h = max(new_h, 10)

    # track_window = (x, y, new_w, new_h)
    
    # --- Resize window according to CAMShift rule ---
    new_w = int(r1 * math.sqrt(m00))
    new_h = int(r2 * math.sqrt(m00))

    x, y, _, _ = track_window
    H, W = frame.shape[:2]

    # ----- PARAMETERS -----
    MIN_SIZE = 12       # prevents collapse-lock
    MAX_SIZE_W = W//2   # prevents explosion
    MAX_SIZE_H = H//2
    # ----------------------

    # --- Clamp size so window stays inside image ---
    if x + new_w > W:
        new_w = W - x
    if y + new_h > H:
        new_h = H - y

    # --- Apply min/max limits ---
    new_w = min(max(new_w, MIN_SIZE), MAX_SIZE_W)
    new_h = min(max(new_h, MIN_SIZE), MAX_SIZE_H)

    track_window = (x, y, new_w, new_h)


    
    # draw
    x, y, w, h = track_window
    result = cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
    cv2.imshow('Tracking', result)

    if cv2.waitKey(30) & 0xFF == 27:
        break

vidCapture.release() 
cv2.destroyAllWindows() 
cv2.waitKey(1)

-1